# OOF ブレンド評価 ― LightGBM(v10) × TCN(v12)

v10(LightGBM)と v12(1D-TCN)の **OOF 予測**を加重平均し、overall RMSE が単独より下がるか確認する。

- OOF はどちらも `artifacts_v10/data/train.parquet` の行順に整列(v10/v12 とも同じparquet基準)。
- target = TVT - last_tvt(相対)。target上RMSE = TVT上RMSE。
- 分割が多少違っても OOF は各行 held-out なのでブレンド評価は妥当。
- 弱いモデルでも系統が違えば(相関が低ければ)ブレンドで改善しうる。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
PARQUET = PROJECT / "v10" / "artifacts_v10" / "data" / "train.parquet"

# 使う OOF(v10 の best と TCN)。必要なら oof_fwd_bwd_cmp.npy に変更。
OOF_LGBM = PROJECT / "artifacts_v10" / "oof_fwd_bwd.npy"
OOF_TCN  = PROJECT / "artifacts_v12" / "oof_tcn.npy"

import pyarrow.parquet as pq
y = pq.read_table(PARQUET, columns=["target"]).column(0).to_numpy().astype(np.float32)
oof_lgbm = np.load(OOF_LGBM).astype(np.float32)
oof_tcn  = np.load(OOF_TCN).astype(np.float32)
assert len(y) == len(oof_lgbm) == len(oof_tcn), (len(y), len(oof_lgbm), len(oof_tcn))
print("rows:", len(y))

FileNotFoundError: C:/Users/kosaka256/Documents/rogii_claude/artifacts_v10/data/train.parquet

In [ ]:
def rmse(p):
    m = np.isfinite(p) & np.isfinite(y)
    return float(np.sqrt(np.mean((p[m] - y[m]) ** 2)))

r_lgbm = rmse(oof_lgbm); r_tcn = rmse(oof_tcn)
print(f"LGBM(v10) overall RMSE = {r_lgbm:.4f}")
print(f"TCN (v12) overall RMSE = {r_tcn:.4f}")
print(f"OOF 相関 corr = {np.corrcoef(oof_lgbm, oof_tcn)[0,1]:.4f}")

In [ ]:
# 重み w を振って blend = w*LGBM + (1-w)*TCN の overall RMSE
ws = np.linspace(0, 1, 51)
scores = [rmse(w*oof_lgbm + (1-w)*oof_tcn) for w in ws]
best_i = int(np.argmin(scores))
w_best = float(ws[best_i]); r_best = float(scores[best_i])

print(f"best w(LGBM)={w_best:.2f}  ->  blend overall RMSE = {r_best:.4f}")
print(f"  LGBM単独 {r_lgbm:.4f} からの改善 Δ = {r_best - r_lgbm:+.4f}")
# いくつか抜粋表示
for w in [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.0]:
    print(f"  w={w:.1f}: {rmse(w*oof_lgbm + (1-w)*oof_tcn):.4f}")